# AMD Hackathon: Vector-based Knowledge Retrieval

This notebook implements the first two phases of our RAG (Retrieval-Augmented Generation) pipeline:
1. **Setup**: Environment configuration and model initialization.
2. **Indexing**: Parsing PDF documents, chunking text, and storing embeddings in ChromaDB.

## Step 1: Environment Setup
In this step, we install the necessary libraries and initialize our embedding model and vector database client.

In [ ]:
!pip install chromadb langchain langchain-community sentence-transformers pymupdf

In [ ]:
import fitz  # PyMuPDF
import chromadb
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB Client
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="design_guides")

print("Setup Complete: ChromaDB and Model initialized.")

## Step 2: Create Chunks and Store in Database
This section handles the document processing pipeline: extraction, splitting, and vector storage.

In [ ]:
def extract_pdf_text(pdf_path):
    """Extracts all text from a PDF file."""
    text = ""
    doc = fitz.open(pdf_path)
    for page in tqdm(doc, desc=f"Reading {pdf_path}"):
        text += page.get_text()
    return text

# Configure Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [ ]:
# List of documents to process
pdfs = [
    "guide 1.pdf",
    "guide 2.pdf",
    "guide 3.pdf"
]

all_chunks = []

for pdf in pdfs:
    print(f"\nProcessing {pdf}")
    text = extract_pdf_text(pdf)
    chunks = splitter.split_text(text)
    
    print(f"Chunks Created: {len(chunks)}")
    
    for chunk in chunks:
        if len(chunk.strip()) > 100:
            all_chunks.append({
                "source": pdf,
                "content": chunk
            })

In [ ]:
# Batch Upload to ChromaDB
batch_size = 32

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Uploading to Chroma"):
    batch = all_chunks[i:i + batch_size]
    texts = [x["content"] for x in batch]
    embeddings = model.encode(texts, show_progress_bar=False)

    collection.add(
        ids=[f"chunk_{i+j}" for j in range(len(batch))],
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=[{"source": x["source"]} for x in batch]
    )

print(f"\nSuccessfully stored {collection.count()} chunks in the database.")

## Step 3: Verification & Testing
Let's verify the storage by performing a sample similarity search.

In [ ]:
# Sample Search
query = "minimum wall thickness"
query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print("=" * 50)
print(f"Search Results for: '{query}'")
print("=" * 50)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1} (Source: {results['metadatas'][0][i]['source']})")
    print("-" * 20)
    print(doc[:500] + "...")

In [ ]:
# Knowledge Base Summary
print("=" * 50)
print("Knowledge Base Summary")
print("=" * 50)
print("PDFs processed:", len(pdfs))
print("Total chunks created:", len(all_chunks))
print("Stored in DB:", collection.count())

## Maintenance & Version Control
Commands for managing the repository.

In [ ]:
!git add .
!git branch -M main
# !git push -u origin main  # Uncomment to push changes